# Use Model Context Protocol (MCP) as tools with Strands Agent

## Overview
The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/introduction) is an open protocol that standardizes how applications provide context to Large Language Models (LLMs). Strands AI SDK integrates with MCP to extend agent capabilities through external tools and services.

MCP enables communication between agents and MCP servers that provide additional tools. The Strands Agent SDK includes built-in support for connecting to MCP servers and using their tools.

In this example we will show you how to use MCP tools on your Strands Agent. We will use the [AWS Documentation MCP server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server/) which provides tools to access AWS documentation, search for content, and get recommendations. This MCP server has 3 main features:

- **Read Documentation**: Fetch and convert AWS documentation pages to markdown format
- **Search Documentation**: Search AWS documentation using the official search API
- **Recommendations**: Get content recommendations for AWS documentation pages



## Agent Details
<div style="float: left; margin-right: 20px;">
    
|Feature             |Description                                        |
|--------------------|---------------------------------------------------|
|Feature used        |MCP Tools                                          |
|Agent Structure     |Single agent architecture                          |

</div>

## Architecture

<div style="text-align:center">
    <img src="images/architecture.png" width="65%" />
</div>

## Key Features
* **Single agent architecture**: this example creates a single agent that interacts with MCP tools
* **MCP tools**: Integration of MCP tools with your agent

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock

Let's now install the requirement packages for our Strands Agent agent

In [ ]:
# installing pre-requisites
!uv pip install -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages

In [3]:
import subprocess
import threading
import time
import sys
import os
from datetime import timedelta

from mcp import StdioServerParameters, stdio_client
from mcp.client.streamable_http import streamablehttp_client
from mcp.server import FastMCP
from strands import Agent
from strands.tools.mcp import MCPClient

# Windows + Jupyter fix: pass DEVNULL as errlog so MCP stdio_client
# doesn't try to use Jupyter's fake stderr which lacks fileno().
import functools
_original_stdio_client = stdio_client
def stdio_client(server, errlog=None):
    return _original_stdio_client(server, errlog=subprocess.DEVNULL)

print('Imports OK - stdio_client patched for Windows/Jupyter')


Imports OK - stdio_client patched for Windows/Jupyter


### Connect to MCP server using stdio transport

[Transports](https://modelcontextprotocol.io/specification/2025-03-26/basic/transports) in MCP provide the foundations for communication between clients and servers. It handles the underlying mechanics of how messages are sent and received. At the moment there are three standards transport implementations built-in in MCP:

- **Standard Input/Output (stdio)**: enables communication through standard input and output streams. It is particularly useful for local integrations and command-line tools
- **Streamable HTTP**: this replaces the HTTP+SSE transport from previous protocol version. In the Streamable HTTP transport, the server operates as an independent process that can handle multiple client connections. This transport uses HTTP POST and GET requests. Server can optionally make use of Server-Sent Events (SSE) to stream multiple server messages. This permits basic MCP servers, as well as more feature-rich servers supporting streaming and server-to-client notifications and requests.
- **SSE**: legacy transport for HTTP-based MCP servers that use Server-Sent Events transport  

Overall, you should use stdio for building command-line tools, implementing local integrations and working with shell scripts. You should use Streamable HTTP transports when you need a flexible and efficient way for AI agents to communicate with tools and services, especially when dealing with stateless communication or when minimizing resource usage is crucial.

You can also use **custom transports** implementation for your specific needs. 


Let's now connect to the MCP server using stdio transport. First of all, we will use the class `MCPClient` to connect to the [AWS Documentation MCP Server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server/). This server provides tools to access AWS documentation, search for content, and get recommendations.

In [4]:
# Connect to an MCP server using stdio transport
stdio_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

#### Setup agent configuration and invoke it

Next we will set our agent configuration using the tools from the `stdio_mcp_client` object we just created. To do so, we need to list the tools available in the MCP server. We can use the `list_tools_sync` method for it. 

After that, we will ask a question to our agent.

In [6]:
# Create an agent with MCP tools
with stdio_mcp_client:
    # Get the tools from the MCP server
    tools = stdio_mcp_client.list_tools_sync()
    
    #List the Tools Names
    for tool in tools:
        print(tool.tool_name)
    print("/n")

    # Create an agent with these tools
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools)

    response = agent("What is Amazon Bedrock pricing model. Be concise.")

print(response)


read_documentation
read_sections
search_documentation
recommend
/n

Tool #1: search_documentation

Tool #2: search_documentation

Tool #3: read_documentation

Tool #4: search_documentation

Tool #5: read_sections

Tool #6: search_documentation
Based on the AWS documentation, here's Amazon Bedrock's pricing model:

## Amazon Bedrock Pricing Model

**Two main pricing options:**

1. **On-Demand (Pay-as-you-go)**
   - Charged per token processed
   - Four token types with different rates:
     - **Input tokens** - tokens in your request
     - **Output tokens** - tokens in the response
     - **Cache read tokens** - reading from prompt cache (cheaper)
     - **Cache write tokens** - writing to prompt cache (more expensive)
   - No upfront commitment required
   - Pricing varies by model and service tier (Reserved, Priority, Standard, Flex)

2. **Provisioned Throughput**
   - Fixed hourly rate for guaranteed capacity
   - Priced by Model Units (MUs) - each MU provides specific throughput le

### Connect to MCP server using Streamable HTTP

Let's now connect to the MCP server using Streamable HTTP transport. First let's start a simple MCP server using Streamable HTTP transport. 

For this example we will create our own MCP server. The architecture will look as following

<div style="text-align:center">
    <img src="images/architecture_2.png" width="65%" />
</div>

In [7]:
# Create an MCP server
mcp = FastMCP("Calculator Server")

# Define a tool


@mcp.tool(description="Calculator tool which performs calculations")
def calculator(x: int, y: int) -> int:
    return x + y


@mcp.tool(description="This is a long running tool")
def long_running_tool(name: str) -> str:
    time.sleep(25)
    return f"Hello {name}"


def main():
    mcp.run(transport="streamable-http", mount_path="mcp")

Let's now start a thread with the `streamable-http` server

In [ ]:
thread = threading.Thread(target=main)
thread.start()

INFO:     Started server process [4740]
INFO:     Waiting for application startup.


[05/26/26 22:06:15] INFO     StreamableHTTP session manager started                  ]8;id=13963335;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\server\streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=13963336;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\server\streamable_http_manager.py#128\128]8;;\

INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


#### Integrating Streamable HTTP client with Agent

Now let's use `streamablehttp_client` integrate this server with a simple agent. 

In [9]:
def create_streamable_http_transport():
    return streamablehttp_client("http://localhost:8000/mcp")


streamable_http_mcp_client = MCPClient(create_streamable_http_transport)

#### Setup agent configuration and invoke it

Next we will set our agent configuration using the tools from the `streamable_http_mcp_client` object we just created. To do so, we need to list the tools available in the MCP server. We can use the `list_tools_sync` method for it. 

After that, we will ask a question to our agent.

In [10]:
with streamable_http_mcp_client:
    tools = streamable_http_mcp_client.list_tools_sync()
    
    #List the Tools Names
    for tool in tools:
        print(tool.tool_name)
    print("/n")

    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools)

    response = str(agent("What is 2 + 2?"))

[05/26/26 22:10:22] INFO     Created new transport with session ID:                  ]8;id=13963342;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\server\streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=13963343;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\server\streamable_http_manager.py#255\255]8;;\
                             755c3a47ac1c48b8aa358979e5b5613b                                                      

INFO:     127.0.0.1:52719 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=13963350;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=13963351;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\httpx\_client.py#1740\1740]8;;\

                    INFO     Received session ID: 755c3a47ac1c48b8aa358979e5b5613b           ]8;id=13963358;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\client\streamable_http.py\streamable_http.py]8;;\:]8;id=13963359;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\client\streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=13963365;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\client\streamable_http.py\streamable_http.py]8;;\:]8;id=13963366;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\client\streamable_http.py#193\193]8;;\

...

2 + 2 = 4

[05/26/26 22:10:27] INFO     Terminating session: 755c3a47ac1c48b8aa358979e5b5613b           ]8;id=13963419;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\server\streamable_http.py\streamable_http.py]8;;\:]8;id=13963420;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\server\streamable_http.py#785\785]8;;\

INFO:     127.0.0.1:52737 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=13963425;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=13963426;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\httpx\_client.py#1740\1740]8;;\

                    INFO     GET stream disconnected, reconnecting in 1000ms...              ]8;id=13963432;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\client\streamable_http.py\streamable_http.py]8;;\:]8;id=13963433;file://c:\Users\batto\Src\NLPtoSQL\.venv\Lib\site-packages\mcp\client\streamable_http.py#298\298]8;;\

### Direct Tool Invocation

While tools are typically invoked by the agent based on user requests, you can also call MCP tools directly. This can be useful for workflow scenarios where you orchestrate multiple tools together.

In [ ]:
query = {"x": 10, "y": 20}

with streamable_http_mcp_client:
    # direct tool invocation
    result = streamable_http_mcp_client.call_tool_sync(
        tool_use_id="tool-123", name="calculator", arguments=query
    )

    # Process the result
    print(f"Calculation result: {result['content'][0]['text']}")

You can optionally also provide `read_timeout_seconds` while calling an MCP server tool to avoid it running for too long

In [ ]:
with streamable_http_mcp_client:
    try:
        result = streamable_http_mcp_client.call_tool_sync(
            tool_use_id="tool-123",
            name="long_running_tool",
            arguments={"name": "Amazon"},
            read_timeout_seconds=timedelta(seconds=30),
        )

        if result["status"] == "error":
            print(f"Tool execution failed: {result['content'][0]['text']}")
        else:
            print(f"Tool execution succeeded: {result['content'][0]['text']}")
    except Exception as e:
        print(f"Tool call timed out or failed: {str(e)}")

### Interacting with multiple MCP servers

With Strands Agents you can also interact with multiple MCP servers using the same agent and configure tools setups such as the max number of tools that can be used in parallel (`max_parallel_tools`). Let's create a new agent to showcase this configuration:

<div style="text-align:center">
    <img src="images/architecture_3.png" width="85%" />
</div>

In this agent, we will again use the AWS Documentation MCP server and we will also use the [AWS IaC MCP Server](https://awslabs.github.io/mcp/servers/aws-iac-mcp-server/) which helps with AWS Cloud Development Kit (CDK) best practices, CloudFormation validation, infrastructure as code patterns and security compliance.

First let's connect to the two MCP servers using the stdio transport

In [ ]:
# Connect to an MCP server using stdio transport
aws_docs_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

# Connect to the AWS IaC MCP server using stdio transport
iac_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(command="uvx", args=["awslabs.aws-iac-mcp-server@latest"])
    )
)


#### Create Agent with MCP servers

Next we will create the agent with the tools from both MCP servers

In [ ]:
# Create an agent with MCP tools
with aws_docs_mcp_client, iac_mcp_client:
    # Get the tools from the MCP server
    tools = aws_docs_mcp_client.list_tools_sync() + iac_mcp_client.list_tools_sync()

    # Create an agent with these tools
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools)

    response = agent(
        "What is Amazon Bedrock pricing model. Be concise. Also what are the best practices related to CDK?"
    )

### Congratulations!

In this notebook you learned how to connect with MCP servers using Strands Agent and two MCP transport protocols: stdio and Streamable HTTP. You also learned how to connect multiple MCP servers to the same agent. Next, let's see how to use different models with your agent